# Fase 2: Obtención y preparación de datos

## Proyecto: Licitaciones desiertas en Chile

### 1. Obtención del dataset

El conjunto de datos utilizado corresponde a los registros de licitaciones públicas publicados por ChileCompra durante el primer semestre de 2026. Los archivos están disponibles de manera mensual, para tener un dataset se anexaron seis archivos mensuales fueron reunidos en un único archivo CSV comprimido en formato ZIP. Debido al tamaño del dataset, el archivo original se almacena en Google Drive y no se incluye en el repositorio GitHub.

In [1]:

from pathlib import Path
import zipfile
import sys

# Identificar la raíz del proyecto
directorio_actual = Path.cwd().resolve()

raiz = next(
    (
        carpeta
        for carpeta in [directorio_actual, *directorio_actual.parents]
        if (carpeta / "requirements.txt").is_file()
        and (carpeta / "F2").is_dir()
    ),
    None
)

if raiz is None:
    raise FileNotFoundError(
        "No se encontró la carpeta principal del proyecto."
    )

# Permitir importar funciones desde src/procesamiento.py
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.procesamiento import (normalizar_montos, consolidar_por_licitacion, comparar_imputaciones)

# Definir la ubicación del dataset
ruta_zip = (
    raiz / "F1" / "data" / "raw"
    / "licitaciones_enero_junio_2026.zip"
)

# Verificar la existencia del archivo
print("Raíz del proyecto:", raiz)
print("¿Existe el dataset?:", ruta_zip.is_file())

if ruta_zip.is_file():
    with zipfile.ZipFile(ruta_zip, "r") as archivo:
        for nombre in archivo.namelist():
            print("Archivo dentro del ZIP:", nombre)
else:
    print("Dataset no encontrado. Será necesario descargarlo.")

print("Funciónes de procesamiento importadas correctamente.")


Raíz del proyecto: C:\Users\Sergio\Desktop\Magister\MCDI500\proyecto-grupo_1-mcdi500
¿Existe el dataset?: True
Archivo dentro del ZIP: licitaciones_enero_junio_2026.csv
Funciónes de procesamiento importadas correctamente.


### 2. Descarga automática del dataset

Para ayudar a la reproducibilidad, se comprobará si el dataset existe en la carpeta de datos originales con el código anterior. Si el archivo está disponible, se utilizará la copia local. En caso contrario, se descargará automáticamente desde Google Drive.

In [2]:
import gdown

# Vemos si el archivo en Google Drive está disponible para poder descargarlo
id_archivo = "1rlzN_3OmS7v3z6Yaiw5jbf3qjrGtuFI1"

# Comprobamos si el dataset ya existe en nuestro proyecto
if ruta_zip.exists():
    print("El dataset ya está disponible.")
else:
    print("Descargando dataset desde Google Drive...")

    ruta_zip.parent.mkdir(parents=True, exist_ok=True)

    gdown.download(
        id=id_archivo,
        output=str(ruta_zip),
        quiet=False
    )

    print("Descarga finalizada.")

# Comprobamos que tenemos el archivo
print("¿Existe el dataset?:", ruta_zip.exists())

El dataset ya está disponible.
¿Existe el dataset?: True


### Validación estructural del archivo CSV

El archivo original contiene registros con cantidades
de campos distintas de las 110 columnas esperadas.

Se generará una copia depurada, conservando solamente
los registros con 110 campos.

El archivo original permanecerá sin modificaciones.
Los registros descartados por problemas estructurales
se contabilizarán y documentarán.

In [3]:

# Validación estructural del CSV original

import csv
import io
import zipfile

# Conservar la ruta del archivo original
ruta_zip_original = ruta_zip

# Archivo depurado, separado de los datos originales
ruta_zip_validado = (
    raiz / "F2" / "data" / "processed"
    / "licitaciones_estructura_valida.zip"
)

ruta_zip_validado.parent.mkdir(
    parents=True,
    exist_ok=True
)

filas_validas = 0
filas_incorrectas = 0
filas_vacias = 0

with zipfile.ZipFile(ruta_zip_original, "r") as origen:

    nombre_csv = origen.namelist()[0]

    with origen.open(nombre_csv) as archivo_original, \
         zipfile.ZipFile(
             ruta_zip_validado,
             "w",
             compression=zipfile.ZIP_DEFLATED
         ) as destino:

        lector_texto = io.TextIOWrapper(
            archivo_original,
            encoding="latin-1",
            newline=""
        )

        lector = csv.reader(
            lector_texto,
            delimiter=";"
        )

        encabezado = next(lector)
        columnas_esperadas = len(encabezado)

        with destino.open(
            "licitaciones_validadas.csv", "w"
        ) as archivo_salida:

            escritor_texto = io.TextIOWrapper(
                archivo_salida,
                encoding="latin-1",
                newline=""
            )

            escritor = csv.writer(
                escritor_texto,
                delimiter=";"
            )

            escritor.writerow(encabezado)

            for fila in lector:

                if len(fila) == columnas_esperadas:
                    escritor.writerow(fila)
                    filas_validas += 1

                elif len(fila) == 0:
                    filas_vacias += 1

                else:
                    filas_incorrectas += 1

            escritor_texto.flush()

print("Columnas esperadas:", columnas_esperadas)
print("Registros válidos:", filas_validas)
print("Registros incorrectos:", filas_incorrectas)
print("Registros vacíos:", filas_vacias)

print("Archivo generado:", ruta_zip_validado)

# Desde aquí, las lecturas posteriores usan
# la copia estructuralmente depurada
ruta_zip = ruta_zip_validado


Columnas esperadas: 110
Registros válidos: 853499
Registros incorrectos: 826
Registros vacíos: 23
Archivo generado: C:\Users\Sergio\Desktop\Magister\MCDI500\proyecto-grupo_1-mcdi500\F2\data\processed\licitaciones_estructura_valida.zip


### Verificación de la estructura depurada

Se verifica que el archivo depurado puede leerse
correctamente y que la columna Estado contiene
valores coherentes con los estados de licitación.

Esta comprobación permite evaluar el efecto de
la depuración estructural.

In [4]:

# Verificar el archivo depurado completo

import pandas as pd

conteo_estados = {}

bloques_verificacion = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    usecols=["CodigoExterno", "Estado"],
    dtype="string",
    chunksize=50000
)

total_verificado = 0

for bloque in bloques_verificacion:
    total_verificado += len(bloque)

    frecuencias = bloque["Estado"].value_counts(
        dropna=False
    )

    for estado, cantidad in frecuencias.items():
        clave = "<NULO>" if pd.isna(estado) else str(estado)

        conteo_estados[clave] = (
            conteo_estados.get(clave, 0) + cantidad
        )

print("Filas verificadas:", total_verificado)

print("\nEstados más frecuentes:")
display(
    pd.Series(conteo_estados)
    .sort_values(ascending=False)
    .head(20)
)


Filas verificadas: 853499

Estados más frecuentes:


Adjudicada                            688955
Cerrada                               112603
Desierta (o art. 3 ó 9 Ley 19.886)     32636
Revocada                               18720
Suspendida                               575
AdjSdicada                                 1
Adjudacada                                 1
0djudicada                                 1
Adjudic(La                                 1
Cerradc                                    1
Adjidicada                                 1
Aajudicada                                 1
Des;1rta (o art. 3 ó 9 Ley 19.886)         1
AdjIdicada                                 1
Adjudi2ada                                 1
dtype: int64

In [5]:

# Leemos las primeras cinco filas del dataset
df = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    nrows=5
)

# Mostramos los primeros registros
display(df.head())

# Revisamos las dimensiones de la muestra
print("Dimensiones:", df.shape)

# Revisamos los tipos de datos
print(df.dtypes)

,Codigo,Link,CodigoExterno,Nombre,Descripcion,CriteriosRequisitosAmbientales,DescripcionCriteriosRequisitosAmbientales,CriteriosRequisitosSociales,DescripcionCriteriosRequisitosSociales,DescripcionCriteriosRequisitosSociales.1,...,Nombre de la Oferta,Estado Oferta,Cantidad Ofertada,Moneda de la Oferta,MontoUnitarioOferta,Valor Total Ofertado,CantidadAdjudicada,MontoLineaAdjudica,FechaEnvioOferta,Oferta seleccionada
0,9482500,http://www.mercadopublico.cl/fichaLicitacion.h...,2277-2-LR25,RENOVACIÓN DE CENTRO COMUNITARIO MONSEÑOR LARRAÍN,Contratación de las obras para la Renovación d...,No,,No,,,...,RENOVACIÓN DE CENTRO COMUNITARIO MONSEÑOR LARRAÍN,Aceptada,1,Peso Chileno,591368040,591368040,0,0,2026-03-06,No Seleccionada
1,9482500,http://www.mercadopublico.cl/fichaLicitacion.h...,2277-2-LR25,RENOVACIÓN DE CENTRO COMUNITARIO MONSEÑOR LARRAÍN,Contratación de las obras para la Renovación d...,No,,No,,,...,Renovacion Centro Comunitario Monseor Larraín,Aceptada,1,Peso Chileno,590294858,590294858,0,0,2026-03-06,No Seleccionada
2,9482500,http://www.mercadopublico.cl/fichaLicitacion.h...,2277-2-LR25,RENOVACIÓN DE CENTRO COMUNITARIO MONSEÑOR LARRAÍN,Contratación de las obras para la Renovación d...,No,,No,,,...,RENOVACIÓN DE CENTRO COMUNITARIO MONSEÑOR LARRAÍN,Aceptada,1,Peso Chileno,539900934,539900934,1,539900934,2026-03-05,Seleccionada
3,9486022,http://www.mercadopublico.cl/fichaLicitacion.h...,799512-11-LE25,CONTRATACIÓN DE SERVICIO DE SEGUROS PARA FLOTA,Las bases están destinadas a fijar los requeri...,No,,No,,,...,Licitacion Seguro de Vehiculos Municipales,Aceptada,1,Peso Chileno,17284291,17284291,1,17284291,2026-01-19,Seleccionada
4,9488213,http://www.mercadopublico.cl/fichaLicitacion.h...,1057417-42-LR25,Suministro insumos y reactivos laboratorio par...,Suministro insumos y reactivos laboratorio par...,No,,No,,,...,Suministro insumos y reactivos laboratorio par...,Aceptada,518741,Peso Chileno,553,286863773,0,0,2026-02-06,No Seleccionada


Dimensiones: (5, 110)
Codigo                  int64
Link                      str
CodigoExterno             str
Nombre                    str
Descripcion               str
                        ...  
Valor Total Ofertado    int64
CantidadAdjudicada      int64
MontoLineaAdjudica      int64
FechaEnvioOferta          str
Oferta seleccionada       str
Length: 110, dtype: object


In [6]:
# Obtenemos una muestra de 1.000 filas
muestra = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    nrows=1000
)

# Vemos cantidad de filas y columnas
print("Filas:", muestra.shape[0])
print("Columnas:", muestra.shape[1])

# Vemos antidad de licitaciones diferentes unicas con el id
print("Licitaciones únicas:", muestra["CodigoExterno"].nunique())

# Vemos cantidad de registros con códigos repetidos
print("Registros repetidos por código:",
      muestra["CodigoExterno"].duplicated().sum())


Filas: 1000
Columnas: 110
Licitaciones únicas: 75
Registros repetidos por código: 925


Esto significa que las primeras 1.000 filas corresponden solamente a 75 códigos de licitación diferentes.

Pero atención: los 925 registros adicionales no necesariamente son errores ni duplicados idénticos.

Una licitación puede tener varias ofertas o ítems asociados, lo que podría explicar que su código aparezca repetido.


In [7]:

# Contamos cuantas veces aparece cada código de licitación
frecuencia = muestra["CodigoExterno"].value_counts()

# Mostramos los 10 códigos más frecuentes
print("Licitaciones con más registros:")
print(frecuencia.head(10))


Licitaciones con más registros:
CodigoExterno
2366-66-LE25        222
1422830-31-LP25      91
1057472-186-LP25     78
886954-283-LE25      74
2410-160-LP25        63
1063535-47-LP25      45
2667-98-LE25         44
1514-145-LP25        28
2292-36-LE25         27
1058095-68-LE25      20
Name: count, dtype: int64


El código 2366-66-LE25 aparece 222 veces dentro de las primeras 1.000 filas, esto significa que esa licitación tiene al menos 222 registros asociados en nuestra muestra.

In [8]:

# Seleccionamos el código más frecuente
codigo = frecuencia.index[0]

# Filtramos los registros de esa licitación
licitacion = muestra[muestra["CodigoExterno"] == codigo]

# Mostramos algunas variables
columnas = [
    "CodigoExterno",
    "Estado",
    "FechaPublicacion",
    "NumeroOferentes"
]

print("Código seleccionado:", codigo)
print("Cantidad de registros:", len(licitacion))

display(licitacion[columnas].head(10))

Código seleccionado: 2366-66-LE25
Cantidad de registros: 222


,CodigoExterno,Estado,FechaPublicacion,NumeroOferentes
228,2366-66-LE25,Adjudicada,2026-01-14,11
229,2366-66-LE25,Adjudicada,2026-01-14,11
230,2366-66-LE25,Adjudicada,2026-01-14,11
231,2366-66-LE25,Adjudicada,2026-01-14,11
232,2366-66-LE25,Adjudicada,2026-01-14,11
233,2366-66-LE25,Adjudicada,2026-01-14,11
234,2366-66-LE25,Adjudicada,2026-01-14,11
235,2366-66-LE25,Adjudicada,2026-01-14,11
236,2366-66-LE25,Adjudicada,2026-01-14,11
237,2366-66-LE25,Adjudicada,2026-01-14,11


### 3. Exploración de estados de licitaciones

identificar los valores existentes en Estado, contar sus apariciones y comprobar si un mismo CodigoExterno está asociado a más de un estado

In [9]:
#identificar los estados presentes en la muestra
print("Estados encontrados:")
print(muestra["Estado"].unique())

Estados encontrados:
<StringArray>
['Cerrada', 'Adjudicada', 'Desierta (o art. 3 ó 9 Ley 19.886)', 'Revocada']
Length: 4, dtype: str


In [10]:
#contar registros (filas) por estado en la muestra  
frecuencia_estados = muestra["Estado"].value_counts(dropna=False)
display(frecuencia_estados)

Estado
Adjudicada                            825
Cerrada                               157
Desierta (o art. 3 ó 9 Ley 19.886)     12
Revocada                                6
Name: count, dtype: int64

In [11]:
#revisar consistencia del estado dentro de cada licitacion de la muestra
estados_por_licitacion = (muestra.groupby("CodigoExterno")["Estado"].nunique(dropna=False))

print("licitaciones con más de un estado:", (estados_por_licitacion > 1).sum())


licitaciones con más de un estado: 0


## 4. Construcción del dataset analítico

El conjunto de datos original contiene múltiples registros
asociados a una misma licitación.

Se preparará una tabla donde cada CodigoExterno represente
una licitación única.

Inicialmente se conservarán las variables relevantes para
el análisis y se comprobará la consistencia de los registros.

In [12]:

# Selección de variables

columnas_analisis = [
    "CodigoExterno",
    "Estado",
    "FechaPublicacion",
    "FechaCierre",
    "NombreOrganismo",
    "sector",
    "RegionUnidad",
    "Tipo",
    "MontoEstimado",
    "Moneda Adquisicion"
]

print("Cantidad de variables:", len(columnas_analisis))

for columna in columnas_analisis:
    print(columna)


Cantidad de variables: 10
CodigoExterno
Estado
FechaPublicacion
FechaCierre
NombreOrganismo
sector
RegionUnidad
Tipo
MontoEstimado
Moneda Adquisicion


In [13]:

# Lectura de las variables seleccionadas

bloques = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    usecols=columnas_analisis,
    dtype={"CodigoExterno": "string"},
    chunksize=50000,
    low_memory=False
)

print("Lectura por bloques preparada correctamente")


Lectura por bloques preparada correctamente


In [14]:

# Comprobar el primer bloque

primer_bloque = next(bloques)

print("Dimensiones del primer bloque:")
print(primer_bloque.shape)

display(primer_bloque.head())


Dimensiones del primer bloque:
(50000, 10)


,CodigoExterno,Estado,NombreOrganismo,sector,RegionUnidad,Tipo,Moneda Adquisicion,FechaCierre,FechaPublicacion,MontoEstimado
0,2277-2-LR25,Cerrada,MUNICIPALIDAD DE PUDAHUEL,Municipalidades,Región Metropolitana de Santiago,LR,Peso Chileno,2026-03-06,2026-01-26,748972966
1,2277-2-LR25,Cerrada,MUNICIPALIDAD DE PUDAHUEL,Municipalidades,Región Metropolitana de Santiago,LR,Peso Chileno,2026-03-06,2026-01-26,748972966
2,2277-2-LR25,Cerrada,MUNICIPALIDAD DE PUDAHUEL,Municipalidades,Región Metropolitana de Santiago,LR,Peso Chileno,2026-03-06,2026-01-26,748972966
3,799512-11-LE25,Adjudicada,CORP MUNICIPAL DE VALPARAISO PARA EL DESARROLL...,Otros,Región de Valparaíso,LE,Peso Chileno,2026-01-19,2026-01-05,4e+07
4,1057417-42-LR25,Cerrada,COMPLEJO ASISTENCIAL DR.VICTOR RIOS RUIZ,Salud,Región del Biobío,LR,Peso Chileno,2026-02-06,2026-01-06,993473792


### Consolidación de registros por licitación

Se procesará el dataset completo por bloques y se construirá
una tabla provisional con una fila por CodigoExterno.

Se comprobarán posibles diferencias entre los estados
de una misma licitación antes de utilizar el resultado.

In [15]:

# Reiniciar la lectura desde el principio

bloques = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    usecols=columnas_analisis,
    dtype={"CodigoExterno": "string"},
    chunksize=50000,
    low_memory=False
)

# Consolidación mediante el módulo reutilizable
(
    df_licitaciones,
    conflictos_estado,
    total_filas,
    codigos_sin_id
) = consolidar_por_licitacion(bloques)

print("Filas originales:", total_filas)
print("Códigos sin identificador:", codigos_sin_id)
print("Licitaciones únicas:", len(df_licitaciones))
print("Licitaciones con múltiples estados:", len(conflictos_estado))

display(df_licitaciones.head())


Filas originales: 853499
Códigos sin identificador: 0
Licitaciones únicas: 46567
Licitaciones con múltiples estados: 10


,CodigoExterno,Estado,NombreOrganismo,sector,RegionUnidad,Tipo,Moneda Adquisicion,FechaCierre,FechaPublicacion,MontoEstimado
0,2277-2-LR25,Cerrada,MUNICIPALIDAD DE PUDAHUEL,Municipalidades,Región Metropolitana de Santiago,LR,Peso Chileno,2026-03-06,2026-01-26,748972966
1,799512-11-LE25,Adjudicada,CORP MUNICIPAL DE VALPARAISO PARA EL DESARROLL...,Otros,Región de Valparaíso,LE,Peso Chileno,2026-01-19,2026-01-05,4e+07
2,1057417-42-LR25,Cerrada,COMPLEJO ASISTENCIAL DR.VICTOR RIOS RUIZ,Salud,Región del Biobío,LR,Peso Chileno,2026-02-06,2026-01-06,993473792
3,2408-54-LP25,Adjudicada,I MUNICIPALIDAD DE LOS ANGELES,Municipalidades,Región del Biobío,LP,Peso Chileno,2026-02-11,2026-01-07,205722589
4,743848-1-LE25,Adjudicada,SEGUNDO TRIBUNAL AMBIENTAL,Legislativo y judicial,Región Metropolitana de Santiago,LE,Peso Chileno,2026-02-02,2026-01-07,5e+07


### Interpretación de los conflictos de estado

Durante la consolidación se identificaron diez códigos de licitación asociados a más de un valor en la columna `Estado`. La revisión permitió determinar que los diez casos presentan un estado reconocido acompañado de un valor no reconocido, aparentemente producto de alteraciones en los datos.

No se identificaron conflictos entre dos estados reconocidos diferentes dentro de estos diez casos. Sin embargo, la presencia de un estado válido no garantiza la integridad del resto de las variables.

Por este motivo, estos registros se mantienen identificados para revisión y se consideran junto con la auditoría posterior de consistencia por licitación. No se realizan correcciones automáticas sobre los valores originales.

In [16]:

# Verificaciones de la consolidación

assert df_licitaciones["CodigoExterno"].is_unique, \
    "Hay códigos duplicados"

assert df_licitaciones["CodigoExterno"].notna().all(), \
    "Hay identificadores vacíos"

assert len(df_licitaciones) <= total_filas, \
    "Se generaron más licitaciones que filas originales"

print("Validación inicial correcta")

print("Dimensiones:", df_licitaciones.shape)


Validación inicial correcta
Dimensiones: (46567, 10)


## 5. Diagnóstico y limpieza de datos

En esta etapa se identifican los valores faltantes,
las inconsistencias y los formatos que requieren corrección.

Se trabaja sobre una copia del dataset analítico,
conservando los datos originales sin modificaciones.

In [17]:

# Crear una copia de trabajo

df_limpio = df_licitaciones.copy()

print("Dimensiones originales:", df_licitaciones.shape)
print("Dimensiones de trabajo:", df_limpio.shape)


Dimensiones originales: (46567, 10)
Dimensiones de trabajo: (46567, 10)


In [18]:

# Diagnóstico de valores faltantes

diagnostico_nulos = pd.DataFrame({
    "Tipo de dato": df_limpio.dtypes.astype(str),
    "Nulos": df_limpio.isna().sum(),
    "Porcentaje (%)": (
        df_limpio.isna().mean() * 100
    ).round(2)
})

display(
    diagnostico_nulos.sort_values(
        "Porcentaje (%)",
        ascending=False
    )
)


,Tipo de dato,Nulos,Porcentaje (%)
sector,str,1628,3.50
MontoEstimado,str,18,0.04
Estado,str,0,0.00
CodigoExterno,str,0,0.00
RegionUnidad,str,0,0.00
NombreOrganismo,str,0,0.00
Tipo,str,0,0.00
Moneda Adquisicion,str,0,0.00
FechaCierre,str,0,0.00
FechaPublicacion,str,0,0.00


In [19]:

# Normalizar columnas de texto

columnas_texto = df_limpio.select_dtypes(
    include=["object", "string"]
).columns

for columna in columnas_texto:
    df_limpio[columna] = (
        df_limpio[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

print("Columnas de texto normalizadas")


Columnas de texto normalizadas


In [20]:

# Revisar los estados de licitación

print("Estados registrados:")

display(
    df_limpio["Estado"].value_counts(
        dropna=False
    )
)


Estados registrados:


Estado
Adjudicada                            35662
Desierta (o art. 3 ó 9 Ley 19.886)     5987
Cerrada                                3603
Revocada                               1300
Suspendida                               15
Name: count, dtype: Int64

### Identificación de estados inconsistentes

Se identificaron valores en la columna Estado que no
corresponden a estados válidos de licitación.

Se analizarán estos registros antes de aplicar
correcciones o eliminaciones.

In [21]:

# Estados válidos observados en el dataset

estados_validos = [
    "Adjudicada",
    "Desierta (o art. 3 ó 9 Ley 19.886)",
    "Cerrada",
    "Revocada",
    "Suspendida"
]

# Identificar registros con estados no reconocidos
mascara_invalidos = (
    df_limpio["Estado"].notna()
    & ~df_limpio["Estado"].isin(estados_validos)
)

registros_invalidos = df_limpio.loc[
    mascara_invalidos
].copy()

print(
    "Registros con estados no reconocidos:",
    len(registros_invalidos)
)

display(
    registros_invalidos[
        ["CodigoExterno", "Estado", "NombreOrganismo", "MontoEstimado"]
    ].head(20)
)


Registros con estados no reconocidos: 0


,CodigoExterno,Estado,NombreOrganismo,MontoEstimado


In [22]:

# Revisar las primeras filas sospechosas con todas sus columnas

display(
    registros_invalidos.head(10).T
)


""
CodigoExterno
Estado
NombreOrganismo
sector
RegionUnidad
Tipo
Moneda Adquisicion
FechaCierre
FechaPublicacion
MontoEstimado


### Separación de registros con problemas estructurales

Durante la exploración se encontraron valores incompatibles
con el significado de algunas columnas.

Se separarán estos registros para su revisión, sin eliminarlos
del archivo original.

In [23]:

# Estados reconocidos en el conjunto de datos

estados_validos = [
    "Adjudicada",
    "Desierta (o art. 3 ó 9 Ley 19.886)",
    "Cerrada",
    "Revocada",
    "Suspendida"
]

# Identificar estados que no corresponden
estado_correcto = df_limpio["Estado"].isin(estados_validos)

# Verificar que exista un código de licitación
codigo_presente = df_limpio["CodigoExterno"].notna()

# Separar los registros
df_sospechosos = df_limpio[
    ~(estado_correcto & codigo_presente)
].copy()

df_revisado = df_limpio[
    estado_correcto & codigo_presente
].copy()

print("Registros originales:", len(df_limpio))
print("Registros que pasan el filtro:", len(df_revisado))
print("Registros para revisión:", len(df_sospechosos))


Registros originales: 46567
Registros que pasan el filtro: 46567
Registros para revisión: 0


In [24]:

# Comprobar que no perdimos registros

assert (
    len(df_revisado) + len(df_sospechosos)
    == len(df_limpio)
)

assert df_revisado["Estado"].isin(
    estados_validos
).all()

print("Separación validada correctamente")

display(
    df_revisado["Estado"].value_counts()
)


Separación validada correctamente


Estado
Adjudicada                            35662
Desierta (o art. 3 ó 9 Ley 19.886)     5987
Cerrada                                3603
Revocada                               1300
Suspendida                               15
Name: count, dtype: Int64

### Definición de la variable objetivo

Para el análisis binario se consideran las licitaciones
con estado Adjudicada o Desierta.

Se define EsDesierta:
- 1: licitación desierta.
- 0: licitación adjudicada.

Los otros estados quedan fuera de este subconjunto analítico,
pero se conservan en los datos de trabajo.

In [25]:

# Seleccionar licitaciones con resultado definido

estados_finales = [
    "Adjudicada",
    "Desierta (o art. 3 ó 9 Ley 19.886)"
]

df_modelo = df_revisado[
    df_revisado["Estado"].isin(estados_finales)
].copy()

# Crear variable binaria
df_modelo["EsDesierta"] = (
    df_modelo["Estado"]
    == "Desierta (o art. 3 ó 9 Ley 19.886)"
).astype(int)

print("Licitaciones para el análisis:", len(df_modelo))

display(
    df_modelo["EsDesierta"].value_counts()
)


Licitaciones para el análisis: 41649


EsDesierta
0    35662
1     5987
Name: count, dtype: int64

### Revisión de variables numéricas

Se revisa la columna MontoEstimado para identificar valores
que no pueden interpretarse como números.

Los valores incompatibles se registran antes de aplicar
las transformaciones de limpieza.

In [26]:

# Conservar el valor original para poder comparar
df_modelo["MontoEstimadoOriginal"] = df_modelo["MontoEstimado"]

# Convertir a número sin modificar todavía el original
df_modelo["MontoEstimadoNumerico"] = pd.to_numeric(
    df_modelo["MontoEstimado"],
    errors="coerce"
)

# Detectar valores presentes que no se pudieron convertir
monto_invalido = (
    df_modelo["MontoEstimado"].notna()
    & df_modelo["MontoEstimadoNumerico"].isna()
)

print("Montos que no se pueden convertir:", monto_invalido.sum())

display(
    df_modelo.loc[
        monto_invalido,
        ["CodigoExterno", "MontoEstimado", "Estado"]
    ].head(15)
)


Montos que no se pueden convertir: 11536


,CodigoExterno,MontoEstimado,Estado
14,1641-308-LR25,"1,04e+09",Adjudicada
21,2258-160-LE25,"6,7e+07",Adjudicada
25,5358-6-LP25,"1,4e+08",Desierta (o art. 3 ó 9 Ley 19.886)
27,1422830-28-LP25,"1,74e+08",Adjudicada
28,2581-25-LE25,"6,2e+07",Desierta (o art. 3 ó 9 Ley 19.886)
35,1057532-69-LP25,"1,44e+08",Adjudicada
43,1514-145-LP25,"1,26e+08",Adjudicada
45,1591-26-LR25,"5,5e+09",Adjudicada
48,1422830-31-LP25,"1,35e+08",Adjudicada
50,4464-58-LR25,"7,7e+09",Adjudicada


### Revisión de variables temporales

Se convierten FechaPublicacion y FechaCierre a formato
datetime para validar su interpretación y calcular
la duración entre ambas fechas.

In [27]:

# Conservar las fechas originales
df_modelo["FechaPublicacionOriginal"] = df_modelo["FechaPublicacion"]
df_modelo["FechaCierreOriginal"] = df_modelo["FechaCierre"]

# Convertir a formato fecha
for columna in ["FechaPublicacion", "FechaCierre"]:
    df_modelo[columna] = pd.to_datetime(
        df_modelo[columna],
        errors="coerce"
    )

# Identificar fechas que no pudieron convertirse
for columna in ["FechaPublicacion", "FechaCierre"]:
    original = df_modelo[columna + "Original"]

    invalidas = (
        original.notna()
        & df_modelo[columna].isna()
    )

    print(columna, "- fechas inválidas:", invalidas.sum())


FechaPublicacion - fechas inválidas: 0
FechaCierre - fechas inválidas: 0


In [28]:

# Calcular los días entre publicación y cierre
df_modelo["DiasPublicacionCierre"] = (
    df_modelo["FechaCierre"]
    - df_modelo["FechaPublicacion"]
).dt.total_seconds() / 86400

# Identificar duraciones negativas
duraciones_negativas = (
    df_modelo["DiasPublicacionCierre"] < 0
)

print(
    "Licitaciones con duración negativa:",
    duraciones_negativas.sum()
)

display(
    df_modelo.loc[
        duraciones_negativas,
        [
            "CodigoExterno",
            "FechaPublicacion",
            "FechaCierre",
            "DiasPublicacionCierre"
        ]
    ].head(10)
)


Licitaciones con duración negativa: 0


,CodigoExterno,FechaPublicacion,FechaCierre,DiasPublicacionCierre


In [29]:

# Comprobaciones del procesamiento

assert "MontoEstimadoNumerico" in df_modelo.columns

assert pd.api.types.is_numeric_dtype(
    df_modelo["MontoEstimadoNumerico"]
)

assert pd.api.types.is_datetime64_any_dtype(
    df_modelo["FechaPublicacion"]
)

assert pd.api.types.is_datetime64_any_dtype(
    df_modelo["FechaCierre"]
)

print("Transformaciones de tipos realizadas correctamente")

display(
    df_modelo[
        [
            "CodigoExterno",
            "Estado",
            "EsDesierta",
            "MontoEstimadoNumerico",
            "DiasPublicacionCierre"
        ]
    ].head(10)
)


Transformaciones de tipos realizadas correctamente


,CodigoExterno,Estado,EsDesierta,MontoEstimadoNumerico,DiasPublicacionCierre
1,799512-11-LE25,Adjudicada,0,40000000.0,14.0
3,2408-54-LP25,Adjudicada,0,205722589.0,35.0
4,743848-1-LE25,Adjudicada,0,50000000.0,26.0
5,1641-157-LR25,Adjudicada,0,1683397800.0,30.0
6,1058095-67-LE25,Adjudicada,0,58438655.0,10.0
7,1058095-68-LE25,Adjudicada,0,58614285.0,12.0
11,1057439-60-LR25,Adjudicada,0,360266050.0,33.0
12,1422825-764-LP25,Adjudicada,0,68012784.0,20.0
14,1641-308-LR25,Adjudicada,0,<NA>,30.0
15,2408-160-LE25,Adjudicada,0,7000000.0,11.0


### Diagnóstico de montos no convertibles

Se identificaron 11.536 registros cuyo MontoEstimado
no pudo convertirse directamente a un valor numérico.

Se analizarán los valores originales para determinar
si corresponden a problemas de formato, información
no disponible o inconsistencias estructurales.

In [30]:

# Analizar los valores originales de los montos inválidos

montos_problematicos = df_modelo.loc[
    monto_invalido,
    "MontoEstimadoOriginal"
]

print("Total de montos problemáticos:", len(montos_problematicos))

print("\nValores más frecuentes:")
display(
    montos_problematicos.value_counts(
        dropna=False
    ).head(20)
)

print("\nEjemplos con otras columnas:")
display(
    df_modelo.loc[
        monto_invalido,
        [
            "CodigoExterno",
            "Estado",
            "MontoEstimadoOriginal",
            "Moneda Adquisicion"
        ]
    ].head(15)
)


Total de montos problemáticos: 11536

Valores más frecuentes:


MontoEstimadoOriginal
1,5e+07    789
2,5e+07    534
1,2e+07    524
6,9e+07    341
1,8e+07    340
3,5e+07    323
1,4e+07    284
1,3e+07    256
4,5e+07    246
1,6e+07    224
1,1e+07    220
1,2e+08    193
3,4e+07    183
2,2e+07    178
2,4e+07    176
6,5e+07    166
1,7e+07    165
3,2e+07    154
5,5e+07    149
1,5e+08    135
Name: count, dtype: Int64


Ejemplos con otras columnas:


,CodigoExterno,Estado,MontoEstimadoOriginal,Moneda Adquisicion
14,1641-308-LR25,Adjudicada,"1,04e+09",Peso Chileno
21,2258-160-LE25,Adjudicada,"6,7e+07",Peso Chileno
25,5358-6-LP25,Desierta (o art. 3 ó 9 Ley 19.886),"1,4e+08",Peso Chileno
27,1422830-28-LP25,Adjudicada,"1,74e+08",Peso Chileno
28,2581-25-LE25,Desierta (o art. 3 ó 9 Ley 19.886),"6,2e+07",Peso Chileno
35,1057532-69-LP25,Adjudicada,"1,44e+08",Peso Chileno
43,1514-145-LP25,Adjudicada,"1,26e+08",Peso Chileno
45,1591-26-LR25,Adjudicada,"5,5e+09",Peso Chileno
48,1422830-31-LP25,Adjudicada,"1,35e+08",Peso Chileno
50,4464-58-LR25,Adjudicada,"7,7e+09",Peso Chileno


### Normalización de montos

Se identificaron montos expresados en notación científica
con coma decimal, por ejemplo 1,5e+07.

Se normaliza el separador decimal para permitir su
conversión numérica, conservando el valor original.

In [31]:

# Normalización definitiva de MontoEstimado

# Identificar los registros que utilizan coma decimal
# Se conserva esta variable para las validaciones posteriores
montos_texto = (
    df_modelo["MontoEstimadoOriginal"]
    .astype("string")
    .str.strip()
)

formato_coma_decimal = montos_texto.str.fullmatch(
    r"[+-]?\d+,\d+([eE][+-]?\d+)?",
    na=False
)

# Utilizar la función del módulo src/procesamiento.py
df_modelo["MontoEstimadoNumerico"] = normalizar_montos(
    df_modelo["MontoEstimadoOriginal"]
)

# Comprobar los resultados
montos_pendientes = df_modelo[
    df_modelo["MontoEstimadoOriginal"].notna()
    & df_modelo["MontoEstimadoNumerico"].isna()
]

print("Montos pendientes:", len(montos_pendientes))

display(
    montos_pendientes["MontoEstimadoOriginal"]
    .value_counts()
    .head(20)
)


Montos pendientes: 0


Series([], Name: count, dtype: Int64)

In [32]:

# Validación de la normalización modularizada

print("Total de licitaciones:", len(df_modelo))

print(
    "Montos originalmente nulos:",
    df_modelo["MontoEstimadoOriginal"].isna().sum()
)

print(
    "Montos no convertibles:",
    (
        df_modelo["MontoEstimadoOriginal"].notna()
        & df_modelo["MontoEstimadoNumerico"].isna()
    ).sum()
)

assert pd.api.types.is_numeric_dtype(
    df_modelo["MontoEstimadoNumerico"]
)

print("Validación correcta")


Total de licitaciones: 41649
Montos originalmente nulos: 7
Montos no convertibles: 0
Validación correcta


In [33]:

# Diagnóstico de montos faltantes según resultado

tabla_faltantes = (
    df_modelo
    .groupby("Estado", dropna=False)
    .agg(
        TotalLicitaciones=("CodigoExterno", "size"),
        MontosFaltantes=(
            "MontoEstimadoNumerico",
            lambda s: s.isna().sum()
        )
    )
)

tabla_faltantes["PorcentajeFaltantes"] = (
    tabla_faltantes["MontosFaltantes"]
    / tabla_faltantes["TotalLicitaciones"]
    * 100
).round(4)

display(tabla_faltantes)


,TotalLicitaciones,MontosFaltantes,PorcentajeFaltantes
Estado,,,
Adjudicada,35662,6,0.0168
Desierta (o art. 3 ó 9 Ley 19.886),5987,1,0.0167


### Evaluación de valores faltantes según resultado de la licitación

Se evaluó la distribución de valores faltantes en `MontoEstimadoNumerico` según el resultado de la licitación, con el propósito de identificar posibles diferencias entre licitaciones adjudicadas y desiertas.

De un total de 41.649 licitaciones, se identificaron siete registros con montos faltantes: seis correspondientes a licitaciones adjudicadas (0,0168 % del grupo) y uno a licitaciones desiertas (0,0167 % del grupo).

Los resultados muestran proporciones prácticamente equivalentes, por lo que no se observa una concentración relevante de valores faltantes en alguno de los dos estados. Sin embargo, esta comprobación no permite afirmar que la ausencia de datos sea completamente aleatoria.

Para el análisis comparativo de montos, se utilizarán los valores originales observados, excluyendo los siete registros sin monto únicamente de dicho análisis. Para el procesamiento orientado a modelos, podrá mantenerse una variable imputada, diferenciándola del monto original y conservando un indicador de imputación.

Esta decisión evita introducir valores estimados en la comparación principal y mantiene la trazabilidad de las transformaciones realizadas.

In [34]:

# Comprobar algunos montos conocidos

ejemplos = df_modelo.loc[
    formato_coma_decimal,
    ["MontoEstimadoOriginal", "MontoEstimadoNumerico"]
].head(10)

display(ejemplos)

assert pd.api.types.is_numeric_dtype(
    df_modelo["MontoEstimadoNumerico"]
)

print("Conversión numérica validada")


,MontoEstimadoOriginal,MontoEstimadoNumerico
14,"1,04e+09",1040000000.0
21,"6,7e+07",67000000.0
25,"1,4e+08",140000000.0
27,"1,74e+08",174000000.0
28,"6,2e+07",62000000.0
35,"1,44e+08",144000000.0
43,"1,26e+08",126000000.0
45,"5,5e+09",5500000000.0
48,"1,35e+08",135000000.0
50,"7,7e+09",7700000000.0


Conversión numérica validada


In [35]:

# Identificar montos que todavía no se pudieron convertir

montos_pendientes = df_modelo[
    df_modelo["MontoEstimadoOriginal"].notna()
    & df_modelo["MontoEstimadoNumerico"].isna()
].copy()

print("Montos pendientes:", len(montos_pendientes))

print("\nValores originales más frecuentes:")
display(
    montos_pendientes["MontoEstimadoOriginal"]
    .value_counts()
    .head(20)
)


Montos pendientes: 0

Valores originales más frecuentes:


Series([], Name: count, dtype: Int64)

In [36]:

# Validación de la normalización

print("Total de registros:", len(df_modelo))

print(
    "Montos convertidos:",
    df_modelo["MontoEstimadoNumerico"].notna().sum()
)

print(
    "Montos faltantes:",
    df_modelo["MontoEstimadoNumerico"].isna().sum()
)

assert pd.api.types.is_numeric_dtype(
    df_modelo["MontoEstimadoNumerico"]
)

print("Tipo numérico validado correctamente")


Total de registros: 41649
Montos convertidos: 41642
Montos faltantes: 7
Tipo numérico validado correctamente


In [37]:

# Revisar los 7 montos faltantes

faltantes_monto = df_modelo[
    df_modelo["MontoEstimadoNumerico"].isna()
].copy()

display(
    faltantes_monto[
        [
            "CodigoExterno",
            "Estado",
            "MontoEstimadoOriginal",
            "Moneda Adquisicion"
        ]
    ]
)

print(
    "Originalmente vacíos:",
    faltantes_monto["MontoEstimadoOriginal"].isna().sum()
)


,CodigoExterno,Estado,MontoEstimadoOriginal,Moneda Adquisicion
30585,1506-5-O126,Adjudicada,<NA>,Peso Chileno
30721,642-10-O126,Desierta (o art. 3 ó 9 Ley 19.886),<NA>,Peso Chileno
31412,1966-6-O126,Adjudicada,<NA>,Peso Chileno
34020,1966-7-O126,Adjudicada,<NA>,Peso Chileno
34045,1966-8-O126,Adjudicada,<NA>,Peso Chileno
37668,5407-47-O126,Adjudicada,<NA>,Peso Chileno
39504,1506-19-O126,Adjudicada,<NA>,Peso Chileno


Originalmente vacíos: 7


### Tratamiento de valores faltantes

La variable MontoEstimadoNumerico presenta 7 valores faltantes
de un total de 41.649 licitaciones.

Se conserva la columna original y se genera una variable
provisional imputada mediante la mediana de los valores
observados. Esta transformación se utilizará para explorar
el preprocesamiento y no constituye todavía el ajuste
definitivo de un modelo predictivo.

In [38]:


# Imputación provisional de montos faltantes

# Seleccionar montos observados en pesos chilenos
mascara_clp = (
    df_modelo["Moneda Adquisicion"] == "Peso Chileno"
)

# Calcular mediana únicamente en CLP
mediana_monto = df_modelo.loc[
    mascara_clp,
    "MontoEstimadoNumerico"
].median()

# Verificar que todos los faltantes sean de CLP
faltantes = df_modelo["MontoEstimadoNumerico"].isna()

assert df_modelo.loc[
    faltantes, "Moneda Adquisicion"
].eq("Peso Chileno").all(), (
    "Hay montos faltantes expresados en otras monedas"
)

# Mantener el monto original normalizado
df_modelo["MontoEstimadoImputado"] = (
    df_modelo["MontoEstimadoNumerico"]
    .fillna(mediana_monto)
)

# Identificar los registros imputados
df_modelo["MontoFueImputado"] = (
    faltantes.astype(int)
)

print("Mediana CLP utilizada:", mediana_monto)
print("Montos imputados:", df_modelo["MontoFueImputado"].sum())
print(
    "Nulos después de imputar:",
    df_modelo["MontoEstimadoImputado"].isna().sum()
)

assert df_modelo["MontoEstimadoImputado"].notna().all()


Mediana CLP utilizada: 27600000.0
Montos imputados: 7
Nulos después de imputar: 0


In [39]:
# Comparación de estrategias de imputación
datos_clp = df_modelo.loc[
    df_modelo["Moneda Adquisicion"] == "Peso Chileno"
].copy()

tabla_imputaciones = comparar_imputaciones(
    datos_clp,
    "MontoEstimadoNumerico"
)

display(tabla_imputaciones)

,estrategia,n,media,desv_est,asimetria,cambio_desv_%,filas_perdidas
0,sin imputar (referencia),40821,1.353832e+08,3.387268e+09,183.175,0.00,7
1,eliminar filas,40821,1.353832e+08,3.387268e+09,183.175,0.00,7
2,media,40828,1.353832e+08,3.386978e+09,183.191,-0.01,0
3,mediana,40828,1.353647e+08,3.386978e+09,183.191,-0.01,0


### Evaluación de estrategias de imputación

Se compararon cuatro estrategias para el tratamiento de valores faltantes en `MontoEstimadoNumerico`, considerando exclusivamente licitaciones cuyos montos están expresados en pesos chilenos.

La muestra contiene 40.828 licitaciones, de las cuales siete presentan montos faltantes. Las estrategias de imputación mediante media y mediana permiten conservar todas las observaciones, mientras que el análisis sin imputación y la eliminación de filas utilizan 40.821 registros con montos observados.

El cambio observado en la desviación estándar al aplicar las estrategias de imputación es aproximadamente de −0,01 % respecto de los registros no imputados. Esto indica un efecto descriptivo reducido sobre dicha medida de dispersión, aunque no descarta efectos sobre otros indicadores.

Se decide mantener `MontoEstimadoNumerico` sin imputar para las comparaciones de montos entre licitaciones adjudicadas y desiertas. Para las transformaciones que requieran valores completos, se conserva `MontoEstimadoImputado`, utilizando la mediana calculada exclusivamente sobre licitaciones expresadas en pesos chilenos y manteniendo `MontoFueImputado` como indicador de trazabilidad.

## 6. Transformación de variables categóricas

Se preparan las variables sector, RegionUnidad y Tipo
para su utilización en análisis estadísticos o modelos.

Se normalizan las categorías y se utiliza One-Hot Encoding
para generar indicadores binarios sin introducir un
orden artificial entre las categorías.

In [40]:

# Seleccionar variables categóricas

columnas_categoricas = [
    "sector",
    "RegionUnidad",
    "Tipo"
]

# Crear una copia para las transformaciones
df_transformado = df_modelo.copy()

# Normalizar texto y completar categorías ausentes
for columna in columnas_categoricas:
    df_transformado[columna] = (
        df_transformado[columna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
        .fillna("No informado")
    )

print("Variables categóricas preparadas")

for columna in columnas_categoricas:
    print(
        columna,
        "- categorías:",
        df_transformado[columna].nunique()
    )


Variables categóricas preparadas
sector - categorías: 8
RegionUnidad - categorías: 16
Tipo - categorías: 12


In [41]:

# Codificación de variables categóricas

df_categorias = pd.get_dummies(
    df_transformado[columnas_categoricas],
    prefix=columnas_categoricas,
    dtype=int
)

print(
    "Columnas creadas por One-Hot Encoding:",
    df_categorias.shape[1]
)

display(df_categorias.head())


Columnas creadas por One-Hot Encoding: 36


,sector_FFAA,"sector_Gob. Central, Universidades",sector_Legislativo y judicial,sector_Municipalidades,sector_No informado,sector_Obras Públicas,sector_Otros,sector_Salud,RegionUnidad_Región Aysén del General Carlos Ibáñez del Campo,RegionUnidad_Región Metropolitana de Santiago,...,Tipo_E2,Tipo_I2,Tipo_L1,Tipo_LE,Tipo_LP,Tipo_LQ,Tipo_LR,Tipo_LS,Tipo_O1,Tipo_O2
1,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
5,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,1,0,0,0
6,0,0,0,0,0,0,0,1,0,1,...,0,0,0,1,0,0,0,0,0,0


In [42]:

# Combinar variables numéricas y categóricas

columnas_numericas = [
    "MontoEstimadoImputado",
    "MontoFueImputado",
    "DiasPublicacionCierre"
]

df_numericas = df_transformado[
    columnas_numericas
].copy()

df_preparado = pd.concat(
    [
        df_numericas,
        df_categorias,
        df_transformado[["EsDesierta"]]
    ],
    axis=1
)

print("Dimensiones:", df_preparado.shape)

display(df_preparado.head())



Dimensiones: (41649, 40)


,MontoEstimadoImputado,MontoFueImputado,DiasPublicacionCierre,sector_FFAA,"sector_Gob. Central, Universidades",sector_Legislativo y judicial,sector_Municipalidades,sector_No informado,sector_Obras Públicas,sector_Otros,...,Tipo_I2,Tipo_L1,Tipo_LE,Tipo_LP,Tipo_LQ,Tipo_LR,Tipo_LS,Tipo_O1,Tipo_O2,EsDesierta
1,40000000.0,0,14.0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
3,205722589.0,0,35.0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,50000000.0,0,26.0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
5,1683397800.0,0,30.0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
6,58438655.0,0,10.0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [43]:

# Validaciones de One-Hot Encoding

assert len(df_preparado) == len(df_modelo), \
    "Se modificó la cantidad de registros"

assert df_preparado["EsDesierta"].isin([0, 1]).all(), \
    "La variable objetivo contiene valores inválidos"

# Cada variable categórica debe activar una categoría por fila
for columna in columnas_categoricas:
    columnas_generadas = [
        c for c in df_categorias.columns
        if c.startswith(columna + "_")
    ]

    assert (
        df_categorias[columnas_generadas].sum(axis=1) == 1
    ).all(), f"Error en la codificación de {columna}"

print("Codificación categórica validada correctamente")

print(
    "Valores faltantes en matriz:",
    df_preparado.isna().sum().sum()
)


Codificación categórica validada correctamente
Valores faltantes en matriz: 0


## 7. Validación del dataset preparado

Se verifica la consistencia de las variables utilizadas
en el análisis, la presencia de valores faltantes,
la integridad de los identificadores y la coherencia
de los datos temporales y numéricos.

Esta auditoría permite identificar problemas pendientes
antes de utilizar el conjunto de datos.

In [44]:

# Validación de identificadores y estados

print("Total de licitaciones:", len(df_modelo))

print(
    "Códigos duplicados:",
    df_modelo["CodigoExterno"].duplicated().sum()
)

print(
    "Códigos faltantes:",
    df_modelo["CodigoExterno"].isna().sum()
)

print(
    "Estados no reconocidos:",
    (~df_modelo["Estado"].isin([
        "Adjudicada",
        "Desierta (o art. 3 ó 9 Ley 19.886)"
    ])).sum()
)

print(
    "Valores faltantes en matriz:",
    df_preparado.isna().sum().sum()
)


Total de licitaciones: 41649
Códigos duplicados: 0
Códigos faltantes: 0
Estados no reconocidos: 0
Valores faltantes en matriz: 0


In [45]:

# Validación de montos y duración

print(
    "Montos negativos:",
    (df_modelo["MontoEstimadoNumerico"] < 0).sum()
)

print(
    "Duraciones negativas:",
    (df_modelo["DiasPublicacionCierre"] < 0).sum()
)

print(
    "Duraciones faltantes:",
    df_modelo["DiasPublicacionCierre"].isna().sum()
)

print(
    "Montos originalmente faltantes:",
    df_modelo["MontoEstimadoNumerico"].isna().sum()
)


Montos negativos: 0
Duraciones negativas: 0
Duraciones faltantes: 0
Montos originalmente faltantes: 7


In [46]:

# Identificar códigos con formato sospechoso

codigos = (
    df_modelo["CodigoExterno"]
    .astype("string")
    .str.strip()
)

# Formato esperado: código de licitación con guiones
formato_codigo = codigos.str.fullmatch(
    r"\d+-\d+-[A-Za-z0-9]+",
    na=False
)

codigos_sospechosos = df_modelo.loc[
    ~formato_codigo
].copy()

print(
    "Códigos con formato sospechoso:",
    len(codigos_sospechosos)
)

display(
    codigos_sospechosos[
        ["CodigoExterno", "Estado", "NombreOrganismo"]
    ].head(15)
)


Códigos con formato sospechoso: 6


,CodigoExterno,Estado,NombreOrganismo
18526,3824-8jer26,Adjudicada,I MUNICIPALIDAD DE PADRE HURTADO
19989,5907- -LE26,Adjudicada,ILUSTRE MUNICIPALIDAD DE HUARA
20670,"t3663-7-L126""",Adjudicada,I MUNICIPALIDAD DE REQUINOA
20685,"10696-2-LE26""",Adjudicada,I MUNICIPALIDAD DE TENO
21573,"92583-3-L126""",Adjudicada,I MUNICIPALIDAD DE MAIPU
22568,"77451-31-R126""",Adjudicada,I MUNICIPALIDAD DE SAN CLEMENTE


## 8. Escalamiento de variables numéricas

Se evalúa el escalamiento de las variables numéricas
para preparar una matriz de características comparable.

Se utiliza StandardScaler como demostración de
estandarización. Para un futuro modelo predictivo,
sus parámetros deberán ajustarse únicamente con
los datos de entrenamiento.

In [47]:

from sklearn.preprocessing import StandardScaler

# Seleccionar variables numéricas continuas
columnas_escalar = [
    "MontoEstimadoImputado",
    "DiasPublicacionCierre"
]

# Crear una copia de la matriz preparada
df_escalado = df_preparado.copy()

# Crear y aplicar el escalador
escalador = StandardScaler()

df_escalado[columnas_escalar] = escalador.fit_transform(
    df_preparado[columnas_escalar]
)

print("Variables escaladas correctamente")

display(
    df_escalado[columnas_escalar].head()
)


Variables escaladas correctamente


,MontoEstimadoImputado,DiasPublicacionCierre
1,-0.027788,-0.009410
3,0.021617,2.859219
4,-0.024806,1.629807
5,0.462134,2.176212
6,-0.022291,-0.555815


In [48]:

# Validación del escalamiento

print("Media de las columnas escaladas:")
display(
    df_escalado[columnas_escalar].mean().round(4)
)

print("Desviación estándar poblacional:")
display(
    df_escalado[columnas_escalar].std(ddof=0).round(4)
)

assert len(df_escalado) == len(df_preparado), \
    "Se modificó la cantidad de registros"

assert df_escalado.isna().sum().sum() == 0, \
    "La matriz contiene valores faltantes"

print("Validación del escalamiento correcta")


Media de las columnas escaladas:


MontoEstimadoImputado   -0.0
DiasPublicacionCierre   -0.0
dtype: float64

Desviación estándar poblacional:


MontoEstimadoImputado    1.0
DiasPublicacionCierre    1.0
dtype: float64

Validación del escalamiento correcta


## 9. Diccionario de variables

Se documentan las variables utilizadas, su significado,
tipo de dato y tratamiento aplicado durante la Fase 2.

In [49]:

# Diccionario de variables

diccionario = pd.DataFrame([
    ["CodigoExterno", "Identificador de licitación", "Texto", "Identificación y revisión de duplicados"],
    ["Estado", "Estado de la licitación", "Categoría", "Selección de resultados definidos"],
    ["EsDesierta", "Indicador de licitación desierta", "Binaria", "1 = Desierta; 0 = Adjudicada"],
    ["FechaPublicacion", "Fecha de publicación", "Fecha", "Conversión a datetime"],
    ["FechaCierre", "Fecha de cierre registrada", "Fecha", "Conversión a datetime"],
    ["DiasPublicacionCierre", "Diferencia entre fechas, en días", "Numérica", "Cálculo a partir de las fechas"],
    ["MontoEstimadoNumerico", "Monto estimado convertido", "Numérica", "Normalización de separadores decimales"],
    ["MontoEstimadoImputado", "Monto con faltantes completados", "Numérica", "Imputación provisional por mediana"],
    ["MontoFueImputado", "Indicador de monto originalmente faltante", "Binaria", "1 = Imputado; 0 = Observado"],
    ["sector", "Sector asociado al registro", "Categoría", "Normalización y One-Hot Encoding"],
    ["RegionUnidad", "Región de la unidad compradora", "Categoría", "Normalización y One-Hot Encoding"],
    ["Tipo", "Tipo de licitación", "Categoría", "Normalización y One-Hot Encoding"]
], columns=[
    "Variable",
    "Descripción",
    "Tipo",
    "Tratamiento"
])

display(diccionario)


,Variable,Descripción,Tipo,Tratamiento
0,CodigoExterno,Identificador de licitación,Texto,Identificación y revisión de duplicados
1,Estado,Estado de la licitación,Categoría,Selección de resultados definidos
2,EsDesierta,Indicador de licitación desierta,Binaria,1 = Desierta; 0 = Adjudicada
3,FechaPublicacion,Fecha de publicación,Fecha,Conversión a datetime
4,FechaCierre,Fecha de cierre registrada,Fecha,Conversión a datetime
5,DiasPublicacionCierre,"Diferencia entre fechas, en días",Numérica,Cálculo a partir de las fechas
6,MontoEstimadoNumerico,Monto estimado convertido,Numérica,Normalización de separadores decimales
7,MontoEstimadoImputado,Monto con faltantes completados,Numérica,Imputación provisional por mediana
8,MontoFueImputado,Indicador de monto originalmente faltante,Binaria,1 = Imputado; 0 = Observado
9,sector,Sector asociado al registro,Categoría,Normalización y One-Hot Encoding


## 10. Exportación de resultados

Se exportan el dataset analítico provisional,
la matriz codificada, la matriz escalada y
el diccionario de variables.

La matriz escalada corresponde a una demostración de
preprocesamiento y no a un conjunto listo para evaluar
modelos predictivos.

In [50]:

from pathlib import Path

# Crear carpeta para los resultados
carpeta_salida = Path("resultados_f2")
carpeta_salida.mkdir(exist_ok=True)

# Guardar los resultados
df_modelo.to_csv(
    carpeta_salida / "licitaciones_analiticas.csv.gz",
    index=False,
    compression="gzip"
)

df_preparado.to_csv(
    carpeta_salida / "matriz_codificada.csv.gz",
    index=False,
    compression="gzip"
)

df_escalado.to_csv(
    carpeta_salida / "matriz_escalada_demostracion.csv.gz",
    index=False,
    compression="gzip"
)

diccionario.to_csv(
    carpeta_salida / "diccionario_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos guardados en:")
print(carpeta_salida.resolve())


Archivos guardados en:
C:\Users\Sergio\Desktop\Magister\MCDI500\proyecto-grupo_1-mcdi500\F2\notebooks\resultados_f2


In [51]:

# Verificación de archivos exportados

archivos = [
    "licitaciones_analiticas.csv.gz",
    "matriz_codificada.csv.gz",
    "matriz_escalada_demostracion.csv.gz",
    "diccionario_variables.csv"
]

for archivo in archivos:
    ruta = carpeta_salida / archivo
    assert ruta.exists(), f"No existe: {archivo}"
    print("OK:", archivo)

# Recuperar la matriz exportada
matriz_recuperada = pd.read_csv(
    carpeta_salida / "matriz_codificada.csv.gz"
)

assert matriz_recuperada.shape == df_preparado.shape

print("\nExportación validada")
print("Dimensiones:", matriz_recuperada.shape)


OK: licitaciones_analiticas.csv.gz
OK: matriz_codificada.csv.gz
OK: matriz_escalada_demostracion.csv.gz
OK: diccionario_variables.csv

Exportación validada
Dimensiones: (41649, 40)


## 11. Revisión de identificadores sospechosos

Durante la validación se identificaron seis licitaciones
cuyos códigos no cumplen el patrón de formato utilizado.

Se revisarán estos registros y sus principales variables
antes de decidir si corresponde excluirlos o corregirlos.

In [52]:

# Examinar identificadores sospechosos

columnas_revision = [
    "CodigoExterno",
    "Estado",
    "NombreOrganismo",
    "FechaPublicacion",
    "FechaCierre",
    "MontoEstimadoOriginal"
]

# Mostrar los códigos completos y sus datos
for _, fila in codigos_sospechosos.iterrows():
    print("\n" + "-" * 50)

    for columna in columnas_revision:
        print(
            columna + ":",
            repr(fila[columna])
        )



--------------------------------------------------
CodigoExterno: '3824-8jer26'
Estado: 'Adjudicada'
NombreOrganismo: 'I MUNICIPALIDAD DE PADRE HURTADO'
FechaPublicacion: Timestamp('2026-03-11 00:00:00')
FechaCierre: Timestamp('2026-03-23 00:00:00')
MontoEstimadoOriginal: '4e+07'

--------------------------------------------------
CodigoExterno: '5907- -LE26'
Estado: 'Adjudicada'
NombreOrganismo: 'ILUSTRE MUNICIPALIDAD DE HUARA'
FechaPublicacion: Timestamp('2026-03-20 00:00:00')
FechaCierre: Timestamp('2026-03-30 00:00:00')
MontoEstimadoOriginal: '31004022'

--------------------------------------------------
CodigoExterno: 't3663-7-L126"'
Estado: 'Adjudicada'
NombreOrganismo: 'I MUNICIPALIDAD DE REQUINOA'
FechaPublicacion: Timestamp('2026-03-24 00:00:00')
FechaCierre: Timestamp('2026-03-30 00:00:00')
MontoEstimadoOriginal: '1e+06'

--------------------------------------------------
CodigoExterno: '10696-2-LE26"'
Estado: 'Adjudicada'
NombreOrganismo: 'I MUNICIPALIDAD DE TENO'
FechaPubl

In [53]:

# Guardar registros sospechosos para su revisión

ruta_anomalias = (
    carpeta_salida / "licitaciones_sospechosas.csv"
)

codigos_sospechosos.to_csv(
    ruta_anomalias,
    index=False,
    encoding="utf-8-sig"
)

assert ruta_anomalias.exists()

print("Registros documentados:", len(codigos_sospechosos))
print("Archivo:", ruta_anomalias.resolve())


Registros documentados: 6
Archivo: C:\Users\Sergio\Desktop\Magister\MCDI500\proyecto-grupo_1-mcdi500\F2\notebooks\resultados_f2\licitaciones_sospechosas.csv


### Separación provisional de identificadores sospechosos

Se identificaron seis códigos de licitación con formatos
no reconocidos por la regla de validación utilizada.

Estos registros se conservan en un archivo independiente
para su verificación contra la fuente original.

Se genera una versión de análisis que los excluye
provisionalmente, sin modificar ni eliminar los registros
del dataset original.

In [54]:

# Separación provisional de los seis códigos

# Crear una máscara utilizando los identificadores sospechosos
mascara_revision = df_modelo["CodigoExterno"].isin(
    codigos_sospechosos["CodigoExterno"]
)

# Conservar el dataset original de trabajo
df_modelo_validado = df_modelo.loc[
    ~mascara_revision
].copy()

# Aplicar el mismo filtro a la matriz preparada
df_preparado_validado = df_preparado.loc[
    df_modelo_validado.index
].copy()

print("Registros antes:", len(df_modelo))
print("Registros separados:", mascara_revision.sum())
print("Registros después:", len(df_modelo_validado))

# Validaciones
assert len(df_modelo_validado) == len(df_preparado_validado)
assert df_modelo_validado["CodigoExterno"].is_unique
assert df_preparado_validado.isna().sum().sum() == 0

print("Separación provisional validada")


Registros antes: 41649
Registros separados: 6
Registros después: 41643
Separación provisional validada


## 12. Validación de consistencia por licitación

Se verifica si una misma licitación presenta valores
diferentes en sus principales variables.

Esta comprobación permite evaluar la validez de
consolidar múltiples registros en una sola fila
por CodigoExterno.

Las discrepancias se documentarán antes de tomar
decisiones de limpieza.

In [55]:

# Auditoría de consistencia del archivo original

columnas_verificar = [
    "Estado",
    "FechaPublicacion",
    "FechaCierre",
    "NombreOrganismo",
    "MontoEstimado"
]

# Almacenar los valores distintos por licitación
valores_por_codigo = {}

bloques_auditoria = pd.read_csv(
    ruta_zip,
    compression="zip",
    sep=";",
    encoding="latin-1",
    usecols=["CodigoExterno"] + columnas_verificar,
    dtype="string",
    chunksize=50000,
    low_memory=False
)

for bloque in bloques_auditoria:

    bloque = bloque.dropna(subset=["CodigoExterno"])

    # Evitar procesar combinaciones repetidas
    for columna in columnas_verificar:

        pares = bloque[
            ["CodigoExterno", columna]
        ].drop_duplicates()

        for codigo, valor in pares.itertuples(
            index=False,
            name=None
        ):

            if codigo not in valores_por_codigo:
                valores_por_codigo[codigo] = {
                    c: set() for c in columnas_verificar
                }

            valor_normalizado = (
                "<NULO>"
                if pd.isna(valor)
                else str(valor).strip()
            )

            valores_por_codigo[codigo][columna].add(
                valor_normalizado
            )

# Identificar discrepancias
resultados_consistencia = {}

for columna in columnas_verificar:

    codigos_conflictivos = [
        codigo
        for codigo, valores in valores_por_codigo.items()
        if len(valores[columna]) > 1
    ]

    resultados_consistencia[columna] = len(
        codigos_conflictivos
    )

print("Licitaciones revisadas:", len(valores_por_codigo))

display(
    pd.DataFrame.from_dict(
        resultados_consistencia,
        orient="index",
        columns=["Licitaciones con diferencias"]
    )
)


Licitaciones revisadas: 46567


,Licitaciones con diferencias
Estado,10
FechaPublicacion,7
FechaCierre,4
NombreOrganismo,22
MontoEstimado,6


In [56]:

# Mostrar ejemplos de licitaciones con diferencias

for columna in columnas_verificar:

    ejemplos = [
        (codigo, sorted(valores[columna]))
        for codigo, valores in valores_por_codigo.items()
        if len(valores[columna]) > 1
    ][:5]

    print(f"\nVariable: {columna}")
    print("Ejemplos con diferencias:")

    for codigo, valores in ejemplos:
        print(codigo, ":", valores)



Variable: Estado
Ejemplos con diferencias:
2445-21-LE26 : ['AdjSdicada', 'Adjudicada']
1213002-6-LE26 : ['Adjudacada', 'Adjudicada']
907562-1-LE26 : ['0djudicada', 'Adjudicada']
1057496-12-LE26 : ['Adjudic(La', 'Adjudicada']
2446-68-LE26 : ['Cerrada', 'Cerradc']

Variable: FechaPublicacion
Ejemplos con diferencias:
2446-58-LE26 : ['2026-03-03', '202URS']
514847-3-LP26 : ['026-03-04"', '2026-03-04']
3017-6-LE26 : ['2026-03-09', '2026-03309']
2446-66-LE26 : ['2026-03-09', '202603-09"']
2446-68-LE26 : ['2026-03-10', '2026-06-10']

Variable: FechaCierre
Ejemplos con diferencias:
1237328-21-LE26 : ['2021;03-26', '2026-03-26']
1057552-8-LE26 : ['2026-03-20', '202T-03-20']
2446-68-LE26 : ['2026-03- 4', '2026-03-24', '2026-03-4"']
729-58-LE26 : ['2026-03-2"', '2026-03-31']

Variable: NombreOrganismo
Ejemplos con diferencias:
1851-8-LE26 : ['MINISTERIO DE OBRAS PUBLICAS DIREC CION GRAL DE OO PP DCYF', 'MINISTERIO DE OBRAS PUBLICAS DIREC CION GRAL DE OO PYFCYF"']
4511-17-LE26 : ['ILUSTRE MUNICI

In [57]:

# Clasificar las diferencias encontradas

resumen_discrepancias = []

for columna in columnas_verificar:

    con_nulos = 0
    sin_nulos = 0

    for codigo, valores in valores_por_codigo.items():

        distintos = valores[columna]

        if len(distintos) > 1:

            if "<NULO>" in distintos:
                con_nulos += 1
            else:
                sin_nulos += 1

    resumen_discrepancias.append({
        "Variable": columna,
        "Diferencias con nulos": con_nulos,
        "Diferencias entre valores": sin_nulos,
        "Total": con_nulos + sin_nulos
    })

resumen_discrepancias = pd.DataFrame(
    resumen_discrepancias
)

display(resumen_discrepancias)


,Variable,Diferencias con nulos,Diferencias entre valores,Total
0,Estado,0,10,10
1,FechaPublicacion,0,7,7
2,FechaCierre,0,4,4
3,NombreOrganismo,0,22,22
4,MontoEstimado,0,6,6


In [58]:

# Revisar licitaciones con distintos estados

estados_conflictivos = {
    codigo: valores["Estado"]
    for codigo, valores in valores_por_codigo.items()
    if len(valores["Estado"]) > 1
}

print(
    "Licitaciones con diferencias de estado:",
    len(estados_conflictivos)
)

for codigo, estados in list(
    estados_conflictivos.items()
)[:15]:
    print(codigo, ":", sorted(estados))


Licitaciones con diferencias de estado: 10
2445-21-LE26 : ['AdjSdicada', 'Adjudicada']
1213002-6-LE26 : ['Adjudacada', 'Adjudicada']
907562-1-LE26 : ['0djudicada', 'Adjudicada']
1057496-12-LE26 : ['Adjudic(La', 'Adjudicada']
2446-68-LE26 : ['Cerrada', 'Cerradc']
1197-9-LE26 : ['Adjidicada', 'Adjudicada']
1216085-10-LP26 : ['Aajudicada', 'Adjudicada']
4844-13-L126 : ['Des;1rta (o art. 3 ó 9 Ley 19.886)', 'Desierta (o art. 3 ó 9 Ley 19.886)']
2446-84-LE26 : ['AdjIdicada', 'Adjudicada']
4451-31-R126 : ['Adjudi2ada', 'Adjudicada']


### Validación de estados reconocidos

Se identificaron valores ajenos a los estados de licitación
dentro de la columna Estado.

Se diferenciarán los códigos con valores no reconocidos
de aquellos que presentan varios estados reconocidos.

Esta revisión no supone que los registros restantes
estén completamente libres de errores estructurales.

In [59]:

# Estados reconocidos durante la exploración

estados_reconocidos = {
    "Adjudicada",
    "Desierta (o art. 3 ó 9 Ley 19.886)",
    "Cerrada",
    "Revocada",
    "Suspendida"
}

# Estados reconocidos encontrados para cada código
estados_validos_por_codigo = {}

for codigo, valores in valores_por_codigo.items():
    estados = valores["Estado"]

    reconocidos = estados.intersection(
        estados_reconocidos
    )

    estados_validos_por_codigo[codigo] = reconocidos

# Separar códigos según cantidad de estados reconocidos
sin_estado_reconocido = {
    codigo: estados
    for codigo, estados in estados_validos_por_codigo.items()
    if len(estados) == 0
}

un_estado_reconocido = {
    codigo: estados
    for codigo, estados in estados_validos_por_codigo.items()
    if len(estados) == 1
}

multiples_estados_reconocidos = {
    codigo: estados
    for codigo, estados in estados_validos_por_codigo.items()
    if len(estados) > 1
}

print(
    "Códigos sin estado reconocido:",
    len(sin_estado_reconocido)
)

print(
    "Códigos con un estado reconocido:",
    len(un_estado_reconocido)
)

print(
    "Códigos con múltiples estados reconocidos:",
    len(multiples_estados_reconocidos)
)



Códigos sin estado reconocido: 0
Códigos con un estado reconocido: 46567
Códigos con múltiples estados reconocidos: 0


### Separación de registros con discrepancias pendientes

Se analizaron 46.567 códigos únicos utilizando el CSV
estructuralmente depurado.

Los 46.567 códigos presentan exactamente un estado reconocido.
No se detectaron códigos con múltiples estados reconocidos.

La auditoría identificó 40 códigos con diferencias en alguna
de las cinco variables revisadas.

Se apartaron provisionalmente 42 registros del subconjunto
analítico, conservando 41.607 licitaciones.

Los registros apartados se mantienen disponibles para revisión.

In [60]:

# Identificar licitaciones con discrepancias

codigos_discrepantes = {
    codigo
    for codigo, valores in valores_por_codigo.items()
    if any(
        len(valores[columna]) > 1
        for columna in columnas_verificar
    )
}

# Incluir los códigos sin estado reconocido
codigos_revision = (
    codigos_discrepantes
    | set(sin_estado_reconocido)
    | set(codigos_sospechosos["CodigoExterno"])
)

# Identificar cuáles aparecen en nuestro dataset analítico
mascara_revision = df_modelo[
    "CodigoExterno"
].isin(codigos_revision)

# Separar registros sin modificar los originales
df_pendientes_revision = df_modelo.loc[
    mascara_revision
].copy()

df_modelo_final_provisional = df_modelo.loc[
    ~mascara_revision
].copy()

# Filtrar también las matrices, conservando los mismos índices
df_preparado_final_provisional = df_preparado.loc[
    df_modelo_final_provisional.index
].copy()

df_escalado_final_provisional = df_escalado.loc[
    df_modelo_final_provisional.index
].copy()

print("Total inicial:", len(df_modelo))
print("Registros apartados:", len(df_pendientes_revision))
print("Registros conservados:", len(df_modelo_final_provisional))
print("Códigos discrepantes detectados:", len(codigos_discrepantes))


Total inicial: 41649
Registros apartados: 42
Registros conservados: 41607
Códigos discrepantes detectados: 40


In [61]:

# Validaciones del subconjunto provisional

assert (
    len(df_modelo_final_provisional)
    + len(df_pendientes_revision)
    == len(df_modelo)
)

assert df_modelo_final_provisional[
    "CodigoExterno"
].is_unique

assert (
    len(df_modelo_final_provisional)
    == len(df_preparado_final_provisional)
    == len(df_escalado_final_provisional)
)

assert (
    df_preparado_final_provisional.isna().sum().sum()
    == 0
)

# Guardar versiones provisionales
df_modelo_final_provisional.to_csv(
    carpeta_salida / "licitaciones_filtradas_provisional.csv.gz",
    index=False,
    compression="gzip"
)

df_preparado_final_provisional.to_csv(
    carpeta_salida / "matriz_filtrada_provisional.csv.gz",
    index=False,
    compression="gzip"
)

df_pendientes_revision.to_csv(
    carpeta_salida / "registros_pendientes_revision.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Validaciones correctas")
print("Archivos provisionales guardados")


Validaciones correctas
Archivos provisionales guardados


## Conclusiones de la Fase 2

### Obtención y exploración

Se trabajó con un conjunto de datos de licitaciones públicas de Chile correspondiente al primer semestre de 2026. Durante la exploración se identificó que una misma licitación podía aparecer en múltiples registros, por lo que se construyó una tabla analítica provisional utilizando `CodigoExterno` como identificador.

### Limpieza de datos

Se detectaron problemas en los formatos de la variable `MontoEstimado`, principalmente números expresados en notación científica y decimales con coma.

Mediante la normalización de estos formatos, se obtuvieron 41.642 montos numéricos sobre un conjunto de 41.649 licitaciones. Los siete valores originalmente faltantes fueron tratados provisionalmente mediante imputación por mediana.

También se identificaron valores anómalos en algunas columnas, incluidos registros con información aparentemente desplazada.

### Transformación de variables

Se construyó la variable binaria `EsDesierta`, donde 1 representa una licitación desierta y 0 una adjudicada.

Se realizó codificación One-Hot Encoding de variables categóricas y se demostró el escalamiento de variables numéricas mediante `StandardScaler`.

La validación del escalamiento obtuvo medias próximas a cero y desviaciones estándar poblacionales próximas a uno.

### Validación y consistencia

El archivo original fue sometido a una validación estructural. Se conservaron 853.499 registros con las 110 columnas esperadas y se apartaron 826 registros con una cantidad incorrecta de campos y 23 registros vacíos.

El archivo depurado permitió identificar 46.567 códigos únicos de licitación. Todos presentaron exactamente un estado reconocido, aunque algunos registros contenían variaciones no reconocidas en ese campo.

La auditoría detectó 40 códigos con diferencias en alguna de las cinco variables examinadas.

Del conjunto analítico inicial de 41.649 licitaciones adjudicadas o desiertas, se apartaron provisionalmente 42 registros.

El subconjunto resultante contiene 41.607 licitaciones, equivalentes aproximadamente al 99,90 % de la población analítica inicial.

### Limitaciones

La consolidación inicial utilizó el primer registro disponible de cada licitación. Aunque posteriormente se apartaron los códigos con discrepancias detectadas, esto no garantiza que todas las variables estén correctamente interpretadas.

Las anomalías observadas en el archivo CSV requieren una comprobación adicional contra la fuente original. Asimismo, la imputación, codificación y estandarización se realizaron como demostraciones de preprocesamiento, no como un pipeline ajustado exclusivamente sobre un conjunto de entrenamiento.

### Resultado

Se obtuvo un conjunto analítico provisional documentado, con variables numéricas y categóricas preparadas, comprobaciones automáticas y archivos exportados. Los registros con discrepancias quedaron separados para su revisión posterior.

In [62]:

# Resumen final de la Fase 2

resumen_final = pd.DataFrame({
    "Indicador": [
        "Licitaciones iniciales",
        "Registros apartados",
        "Licitaciones conservadas",
        "Porcentaje conservado (%)",
        "Códigos discrepantes en origen",
        "Valores faltantes en matriz filtrada",
        "Identificadores duplicados"
    ],
    "Resultado": [
        len(df_modelo),
        len(df_pendientes_revision),
        len(df_modelo_final_provisional),
        round(
            len(df_modelo_final_provisional)
            / len(df_modelo) * 100, 2
        ),
        len(codigos_discrepantes),
        df_preparado_final_provisional.isna().sum().sum(),
        df_modelo_final_provisional[
            "CodigoExterno"
        ].duplicated().sum()
    ]
})

display(resumen_final)

# Exportar el resumen
resumen_final.to_csv(
    carpeta_salida / "resumen_final_f2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resumen exportado correctamente")


,Indicador,Resultado
0,Licitaciones iniciales,41649.0
1,Registros apartados,42.0
2,Licitaciones conservadas,41607.0
3,Porcentaje conservado (%),99.9
4,Códigos discrepantes en origen,40.0
5,Valores faltantes en matriz filtrada,0.0
6,Identificadores duplicados,0.0


Resumen exportado correctamente
